In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
import optuna
import joblib

In [2]:
df = pd.read_csv(r'..\..\..\data\002_processed_data\xy_df.csv')
df = df.dropna(subset=['stability'])

exclude_cols = ['smiles', 'homo', 'lumo', 'stability']
feature_cols = [col for col in df.columns if col not in exclude_cols]

X = df[feature_cols].copy()
y = df['stability'].copy()

print(f"   Dataset: {len(df)} molecules")
print(f"   Features: {len(feature_cols)}")

   Dataset: 310 molecules
   Features: 217


In [3]:
X = X.fillna(0)
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

# Clip extremes
numeric_cols = X.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    q99 = X[col].quantile(0.99)
    q01 = X[col].quantile(0.01)
    if q99 > q01:
        X[col] = X[col].clip(lower=q01, upper=q99)

# Remove constant features
constant_features = X.columns[X.std() == 0].tolist()
if len(constant_features) > 0:
    X = X.drop(columns=constant_features)
    feature_cols = [col for col in feature_cols if col not in constant_features]

# Encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y)


In [4]:
def extract_bond_atom_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {f: 0 for f in ['n_bonds', 'n_single_bonds', 'n_double_bonds',
                'n_triple_bonds', 'n_aromatic_bonds', 'count_c', 'count_h',
                'count_o', 'count_n', 'count_s', 'count_f', 'count_cl',
                'count_br', 'count_i', 'count_p', 'count_si']}
    
    bonds = mol.GetBonds()
    atoms = mol.GetAtoms()
    atom_symbols = [atom.GetSymbol() for atom in atoms]
    
    return {
        'n_bonds': len(bonds),
        'n_single_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 1.0),
        'n_double_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 2.0),
        'n_triple_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 3.0),
        'n_aromatic_bonds': sum(1 for b in bonds if b.GetBondTypeAsDouble() == 1.5),
        'count_c': atom_symbols.count('C'),
        'count_h': atom_symbols.count('H'),
        'count_o': atom_symbols.count('O'),
        'count_n': atom_symbols.count('N'),
        'count_s': atom_symbols.count('S'),
        'count_f': atom_symbols.count('F'),
        'count_cl': atom_symbols.count('Cl'),
        'count_br': atom_symbols.count('Br'),
        'count_i': atom_symbols.count('I'),
        'count_p': atom_symbols.count('P'),
        'count_si': atom_symbols.count('Si')
    }

bond_atom_features = df['smiles'].apply(extract_bond_atom_features)
bond_atom_df = pd.DataFrame(bond_atom_features.tolist())

print(f"   Bond/atom features: {bond_atom_df.shape}")

X_reset = X.reset_index(drop=True)
bond_reset = bond_atom_df.reset_index(drop=True)

#If there are any bond features, we remove them to prevent duplicates
existing_cols = set(X_reset.columns)
new_bond_cols = [col for col in bond_reset.columns if col not in existing_cols]

if len(new_bond_cols) < len(bond_reset.columns):
    print(f"   Note: {len(bond_reset.columns) - len(new_bond_cols)} bond features already exist in X")
    bond_reset = bond_reset[new_bond_cols]

X = pd.concat([X_reset, bond_reset], axis=1)

# Verify no duplicates
assert not X.columns.duplicated().any(), "Duplicate columns detected!"

print(f"   Combined features: {X.shape[1]}")



   Bond/atom features: (310, 16)
   Combined features: 189


In [5]:
#interaction features:



X['TPSA_x_MolLogP'] = X['TPSA'] * X['MolLogP']
X['aromatic_x_hetero'] = X['NumAromaticRings'] * X['NumHeteroatoms']
X['bonds_x_carbon'] = X['n_bonds'] * X['count_c']
X['aromatic_bonds_x_carbon'] = X['n_aromatic_bonds'] * X['count_c']


print(f"   Total: {X.shape[1]} features")

#chem.-infromed features:

X['pi_electrons'] = X['n_aromatic_bonds'] * 2 + X['n_double_bonds'] * 2
X['conjugation_score'] = (X['NumAromaticRings'] * X['n_aromatic_bonds']) / (X['HeavyAtomCount'] + 1)
X['aromatic_ratio'] = X['NumAromaticRings'] / (X['RingCount'] + 1)
X['aromatic_atoms_ratio'] = (X['n_aromatic_bonds'] * 2) / (X['HeavyAtomCount'] + 1)
X['aromatic_x_double'] = X['n_aromatic_bonds'] * X['n_double_bonds']

# Conjugation features



X['hetero_ratio'] = X['NumHeteroatoms'] / (X['HeavyAtomCount'] + 1)
X['hetero_in_aromatic'] = X['NumAromaticHeterocycles'] / (X['NumAromaticRings'] + 1)
X['nitrogen_ratio'] = X['count_n'] / (X['HeavyAtomCount'] + 1)
X['oxygen_ratio'] = X['count_o'] / (X['HeavyAtomCount'] + 1)
X['sulfur_ratio'] = X['count_s'] / (X['HeavyAtomCount'] + 1)

  # Heteroatom: 5 features



# Electron-withdrawing groups
ewg_components = [
    X['count_f'],
    X['count_cl'],
    X['count_br'],
    X.get('fr_nitro', pd.Series(0, index=X.index)),
    X.get('fr_nitro_arom', pd.Series(0, index=X.index))
]
X['EWG_count'] = pd.concat(ewg_components, axis=1).sum(axis=1)
X['EWG_density'] = X['EWG_count'] / (X['HeavyAtomCount'] + 1)

# Electron-donating groups (use .get() for safety)
edg_components = [
    X.get('fr_NH2', pd.Series(0, index=X.index)),
    X.get('fr_NH1', pd.Series(0, index=X.index)),
    X.get('fr_NH0', pd.Series(0, index=X.index)),
    X.get('fr_methoxy', pd.Series(0, index=X.index)),
    X.get('fr_Ar_NH', pd.Series(0, index=X.index))
]
X['EDG_count'] = pd.concat(edg_components, axis=1).sum(axis=1)
X['EDG_density'] = X['EDG_count'] / (X['HeavyAtomCount'] + 1)
X['EWG_EDG_balance'] = (X['EDG_count'] - X['EWG_count']) / (X['HeavyAtomCount'] + 1)

# EWG/EDG: 5 features



X['TPSA_per_atom'] = X['TPSA'] / (X['HeavyAtomCount'] + 1)
X['TPSA_per_MW'] = X['TPSA'] / (X['MolWt'] + 1)
X['MolMR_per_atom'] = X['MolMR'] / (X['HeavyAtomCount'] + 1)
X['LogP_per_atom'] = X['MolLogP'] / (X['HeavyAtomCount'] + 1)

#Size-normalized: 4 features



X['charge_range'] = X['MaxPartialCharge'] - X['MinPartialCharge']
X['charge_extent'] = X['MaxAbsPartialCharge'] - X['MinAbsPartialCharge']
X['charge_asymmetry'] = abs(X['MaxPartialCharge'] + X['MinPartialCharge'])

# Charge: 3 features



X['unsaturation'] = (X['n_double_bonds'] + X['n_triple_bonds'] + X['n_aromatic_bonds']) / (X['n_bonds'] + 1)
X['rigidity'] = X['RingCount'] / (X['NumRotatableBonds'] + 1)
X['ring_complexity'] = (X['NumAromaticRings'] + X['NumSaturatedRings'] + X['NumAliphaticRings']) / (X['HeavyAtomCount'] + 1)
X['heterocycle_ratio'] = X['NumHeterocycles'] / (X['RingCount'] + 1)

#Complexity: 4 features



X['aromatic_polarity'] = X['NumAromaticRings'] * X['TPSA']
X['hetero_conjugation'] = X['NumHeteroatoms'] * X['n_aromatic_bonds']
X['nitrogen_aromatic'] = X['count_n'] * X['NumAromaticRings']
X['oxygen_aromatic'] = X['count_o'] * X['NumAromaticRings']
X['hetero_per_ring'] = X['NumHeteroatoms'] / (X['RingCount'] + 1)
X['aromatic_hetero_ratio'] = X['NumAromaticHeterocycles'] / (X['NumHeteroatoms'] + 1)

#Orbital interactions: 6 features



X['single_bond_ratio'] = X['n_single_bonds'] / (X['n_bonds'] + 1)
X['multiple_bond_ratio'] = (X['n_double_bonds'] + X['n_triple_bonds']) / (X['n_bonds'] + 1)
X['aromatic_bond_ratio'] = X['n_aromatic_bonds'] / (X['n_bonds'] + 1)

#Bonding: 3 features



X = X.fillna(0)
feature_cols = X.columns.tolist()

print(f"\n otal chemistry features added: 35")
print(f"  Total features now: {len(feature_cols)}")
print(f"shape: {X.shape}")



   Total: 193 features

 otal chemistry features added: 35
  Total features now: 228
shape: (310, 228)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"   Train: {X_train.shape[0]} samples")
print(f"   Test: {X_test.shape[0]} samples")

   Train: 248 samples
   Test: 62 samples


In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"   After: {pd.Series(y_train_balanced).value_counts().to_dict()}")

   After: {2: 113, 1: 113, 0: 113}


In [8]:
def objective(trial):
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet'])
    params = {
        'C': trial.suggest_float('C', 0.01, 100.0, log=True),
        'penalty': penalty,
        'solver': 'saga',
        'max_iter': 2000,
        'class_weight': 'balanced',
        'random_state': 42
    }
    if penalty == 'elasticnet':
        params['l1_ratio'] = trial.suggest_float('l1_ratio', 0, 1)
    
    model = LogisticRegression(**params)
    scores = cross_val_score(model, X_train_balanced, y_train_balanced,
                            cv=5, scoring='f1_weighted', n_jobs=-1)
    return scores.mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\n   Best params: {study.best_params}")
print(f"   Best CV F1: {study.best_value:.4f}")

# Train final LR with best params
best_params = study.best_params.copy()
best_params['solver'] = 'saga'
best_params['max_iter'] = 2000
best_params['class_weight'] = 'balanced'
best_params['random_state'] = 42

if best_params.get('penalty') != 'elasticnet' and 'l1_ratio' in best_params:
    del best_params['l1_ratio']

lr_model = LogisticRegression(**best_params)
lr_model.fit(X_train_balanced, y_train_balanced)

# Evaluate LR
y_pred_lr = lr_model.predict(X_test_scaled)
lr_f1 = f1_score(y_test, y_pred_lr, average='weighted')
lr_acc = accuracy_score(y_test, y_pred_lr)

print(f"\n   Logistic Regression Test F1: {lr_f1:.4f}")

Best trial: 14. Best value: 0.931245: 100%|██████████| 100/100 [05:28<00:00,  3.29s/it]



   Best params: {'penalty': 'l1', 'C': 0.40375562311868335}
   Best CV F1: 0.9312

   Logistic Regression Test F1: 0.9514


In [9]:
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_balanced, y_train_balanced)

y_pred_rf = rf_model.predict(X_test_scaled)
rf_f1 = f1_score(y_test, y_pred_rf, average='weighted')
rf_acc = accuracy_score(y_test, y_pred_rf)

In [10]:
voting_clf = VotingClassifier(
    estimators=[
        ('lr', lr_model),
        ('rf', rf_model)
    ],
    voting='soft',  # using soft voting so we dont predict based on pure votes but probabilities
    weights=[2, 1],  
    n_jobs=-1
)

voting_clf.fit(X_train_balanced, y_train_balanced)

y_pred_voting = voting_clf.predict(X_test_scaled)
voting_f1 = f1_score(y_test, y_pred_voting, average='weighted')
voting_acc = accuracy_score(y_test, y_pred_voting)

print(f"   Voting Ensemble Test F1: {voting_f1:.4f}")

   Voting Ensemble Test F1: 0.9514


In [11]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Voting Ensemble'],
    'Test F1': [lr_f1, rf_f1, voting_f1],
    'Test Accuracy': [lr_acc, rf_acc, voting_acc]
}).sort_values('Test F1', ascending=False)

print(comparison.to_string(index=False))

# Select best model
best_name = comparison.iloc[0]['Model']
best_f1 = comparison.iloc[0]['Test F1']

if 'Voting' in best_name:
    final_model = voting_clf
elif 'Random' in best_name:
    final_model = rf_model
else:
    final_model = lr_model

print(f"\nBest Model: {best_name}")
print(f"   Test F1: {best_f1:.4f}")

# Detailed evaluation
print(f"\nClassification Report ({best_name}):")
if 'Voting' in best_name:
    y_pred_final = y_pred_voting
elif 'Random' in best_name:
    y_pred_final = y_pred_rf
else:
    y_pred_final = y_pred_lr

print(classification_report(y_test, y_pred_final, target_names=le.classes_))

              Model  Test F1  Test Accuracy
Logistic Regression 0.951403       0.951613
    Voting Ensemble 0.951403       0.951613
      Random Forest 0.935211       0.935484

Best Model: Logistic Regression
   Test F1: 0.9514

Classification Report (Logistic Regression):
              precision    recall  f1-score   support

        high       0.94      1.00      0.97        17
         low       0.94      0.94      0.94        17
      medium       0.96      0.93      0.95        28

    accuracy                           0.95        62
   macro avg       0.95      0.96      0.95        62
weighted avg       0.95      0.95      0.95        62



In [12]:
import joblib


# Save everything
joblib.dump(final_model, r'algor_used\classification_lr_model.joblib')
joblib.dump(scaler, r'algor_used\classification_scaler.joblib')
feature_names = X_train.columns.tolist()
joblib.dump(feature_names, r'algor_used\classification_feature_names.joblib')
joblib.dump(le,r'algor_used\label_encoder_classification.joblib')
print(f" Saved: feature_names ({len(feature_names)} features)")



 Saved: feature_names (228 features)


SUBMISSION

In [13]:
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
import warnings

kaggle_test_df = pd.read_csv(r'..\..\..\data\003_kaggle_competition_data\25261_classification_sample_submission.csv')
kaggle_smiles = kaggle_test_df[['smiles']].copy()

In [15]:
import joblib
import numpy as np
import pandas as pd

# Load 
final_model = joblib.load(r'algor_used\classification_lr_model.joblib')
scaler = joblib.load(r'algor_used\classification_scaler.joblib')
feature_cols = joblib.load(r'algor_used\classification_feature_names.joblib')
le = joblib.load(r'algor_used\label_encoder_classification.joblib')
print(f"Loaded: feature_names ({len(feature_cols)} features)")

Loaded: feature_names (228 features)


In [16]:
descriptor_names = [desc[0] for desc in Descriptors._descList]
descriptor_calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)

def compute_descriptors_fast(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [np.nan] * len(descriptor_names)
    try:
        return list(descriptor_calculator.CalcDescriptors(mol))
    except:
        return [np.nan] * len(descriptor_names)

test_descriptors = []
for idx, smiles in enumerate(kaggle_test_df['smiles']):
    if idx % 50 == 0:
        print(f"   Progress: {idx}/{len(kaggle_test_df)}...")
    test_descriptors.append(compute_descriptors_fast(smiles))

X_kaggle = pd.DataFrame(test_descriptors, columns=descriptor_names)
print(f"  Descriptors computed: {X_kaggle.shape}")

   Progress: 0/200...
   Progress: 50/200...
   Progress: 100/200...
   Progress: 150/200...
  Descriptors computed: (200, 217)


In [17]:

bond_features = [extract_bond_atom_features(s) for s in kaggle_test_df['smiles']]
bond_df = pd.DataFrame(bond_features)

X_kaggle = pd.concat([X_kaggle, bond_df], axis=1)
X_kaggle = X_kaggle.loc[:, ~X_kaggle.columns.duplicated()]

print(f"   Bond/atom features: {bond_df.shape}")
print(f"   Combined: {X_kaggle.shape}")

   Bond/atom features: (200, 16)
   Combined: (200, 233)


In [18]:
X_kaggle['TPSA_x_MolLogP'] = X_kaggle['TPSA'] * X_kaggle['MolLogP']
X_kaggle['aromatic_x_hetero'] = X_kaggle['NumAromaticRings'] * X_kaggle['NumHeteroatoms']
X_kaggle['bonds_x_carbon'] = X_kaggle['n_bonds'] * X_kaggle['count_c']
X_kaggle['aromatic_bonds_x_carbon'] = X_kaggle['n_aromatic_bonds'] * X_kaggle['count_c']

In [19]:
X_kaggle['pi_electrons'] = X_kaggle['n_aromatic_bonds'] * 2 + X_kaggle['n_double_bonds'] * 2
X_kaggle['conjugation_score'] = (X_kaggle['NumAromaticRings'] * X_kaggle['n_aromatic_bonds']) / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['aromatic_ratio'] = X_kaggle['NumAromaticRings'] / (X_kaggle['RingCount'] + 1)
X_kaggle['aromatic_atoms_ratio'] = (X_kaggle['n_aromatic_bonds'] * 2) / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['aromatic_x_double'] = X_kaggle['n_aromatic_bonds'] * X_kaggle['n_double_bonds']

# Heteroatom features (5)
X_kaggle['hetero_ratio'] = X_kaggle['NumHeteroatoms'] / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['hetero_in_aromatic'] = X_kaggle['NumAromaticHeterocycles'] / (X_kaggle['NumAromaticRings'] + 1)
X_kaggle['nitrogen_ratio'] = X_kaggle['count_n'] / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['oxygen_ratio'] = X_kaggle['count_o'] / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['sulfur_ratio'] = X_kaggle['count_s'] / (X_kaggle['HeavyAtomCount'] + 1)

# EWG/EDG features (5)
X_kaggle['EWG_count'] = X_kaggle['count_f'] + X_kaggle['count_cl'] + X_kaggle['count_br'] + X_kaggle['fr_nitro'] + X_kaggle['fr_nitro_arom']
X_kaggle['EWG_density'] = X_kaggle['EWG_count'] / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['EDG_count'] = X_kaggle['fr_NH2'] + X_kaggle['fr_NH1'] + X_kaggle['fr_NH0'] + X_kaggle['fr_methoxy'] + X_kaggle['fr_Ar_NH']
X_kaggle['EDG_density'] = X_kaggle['EDG_count'] / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['EWG_EDG_balance'] = (X_kaggle['EDG_count'] - X_kaggle['EWG_count']) / (X_kaggle['HeavyAtomCount'] + 1)

# Size-normalized properties (4)
X_kaggle['TPSA_per_atom'] = X_kaggle['TPSA'] / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['TPSA_per_MW'] = X_kaggle['TPSA'] / (X_kaggle['MolWt'] + 1)
X_kaggle['MolMR_per_atom'] = X_kaggle['MolMR'] / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['LogP_per_atom'] = X_kaggle['MolLogP'] / (X_kaggle['HeavyAtomCount'] + 1)

# Charge distribution (3)
X_kaggle['charge_range'] = X_kaggle['MaxPartialCharge'] - X_kaggle['MinPartialCharge']
X_kaggle['charge_extent'] = X_kaggle['MaxAbsPartialCharge'] - X_kaggle['MinAbsPartialCharge']
X_kaggle['charge_asymmetry'] = abs(X_kaggle['MaxPartialCharge'] + X_kaggle['MinPartialCharge'])

# Complexity/rigidity (4)
X_kaggle['unsaturation'] = (X_kaggle['n_double_bonds'] + X_kaggle['n_triple_bonds'] + X_kaggle['n_aromatic_bonds']) / (X_kaggle['n_bonds'] + 1)
X_kaggle['rigidity'] = X_kaggle['RingCount'] / (X_kaggle['NumRotatableBonds'] + 1)
X_kaggle['ring_complexity'] = (X_kaggle['NumAromaticRings'] + X_kaggle['NumSaturatedRings'] + X_kaggle['NumAliphaticRings']) / (X_kaggle['HeavyAtomCount'] + 1)
X_kaggle['heterocycle_ratio'] = X_kaggle['NumHeterocycles'] / (X_kaggle['RingCount'] + 1)

# Orbital-specific interactions (6)
X_kaggle['aromatic_polarity'] = X_kaggle['NumAromaticRings'] * X_kaggle['TPSA']
X_kaggle['hetero_conjugation'] = X_kaggle['NumHeteroatoms'] * X_kaggle['n_aromatic_bonds']
X_kaggle['nitrogen_aromatic'] = X_kaggle['count_n'] * X_kaggle['NumAromaticRings']
X_kaggle['oxygen_aromatic'] = X_kaggle['count_o'] * X_kaggle['NumAromaticRings']
X_kaggle['hetero_per_ring'] = X_kaggle['NumHeteroatoms'] / (X_kaggle['RingCount'] + 1)
X_kaggle['aromatic_hetero_ratio'] = X_kaggle['NumAromaticHeterocycles'] / (X_kaggle['NumHeteroatoms'] + 1)

# Bonding character (3)
X_kaggle['single_bond_ratio'] = X_kaggle['n_single_bonds'] / (X_kaggle['n_bonds'] + 1)
X_kaggle['multiple_bond_ratio'] = (X_kaggle['n_double_bonds'] + X_kaggle['n_triple_bonds']) / (X_kaggle['n_bonds'] + 1)
X_kaggle['aromatic_bond_ratio'] = X_kaggle['n_aromatic_bonds'] / (X_kaggle['n_bonds'] + 1)

X_kaggle = X_kaggle.fillna(0)

print(f"   Added 35 chemistry features to Kaggle data")
print(f"   Total: {X_kaggle.shape}")

   Added 35 chemistry features to Kaggle data
   Total: (200, 272)


In [20]:
X_kaggle = X_kaggle.replace([np.inf, -np.inf], 0)

# Clip extremes
for col in X_kaggle.select_dtypes(include=[np.number]).columns:
    q99 = X_kaggle[col].quantile(0.99)
    q01 = X_kaggle[col].quantile(0.01)
    if q99 > q01:
        X_kaggle[col] = X_kaggle[col].clip(lower=q01, upper=q99)

# DON'T remove constant features!
print(f"   After preprocessing: {X_kaggle.shape}")

   After preprocessing: (200, 272)


In [21]:
missing_features = [feat for feat in feature_cols if feat not in X_kaggle.columns]

if missing_features:
    print(f"Adding {len(missing_features)} missing features as zeros")
    for feat in missing_features:
        X_kaggle[feat] = 0
else:
    print(f"All {len(feature_cols)} features present!")

X_kaggle = X_kaggle[feature_cols]

print(f"   Final shape: {X_kaggle.shape}")
assert X_kaggle.shape[1] == len(feature_cols), "Feature count mismatch!"

All 228 features present!
   Final shape: (200, 228)


In [22]:
X_kaggle_scaled = scaler.transform(X_kaggle)
print(f"Scaled: {X_kaggle_scaled.shape}")

Scaled: (200, 228)


In [23]:
predictions_encoded = final_model.predict(X_kaggle_scaled)
predictions = le.inverse_transform(predictions_encoded)

print(f"Predictions: {len(predictions)}")

# Distribution
pred_counts = pd.Series(predictions).value_counts()
print(f"\n   Prediction distribution:")
for cls in ['high', 'low', 'medium']:
    count = pred_counts.get(cls, 0)
    pct = count / len(predictions) * 100

Predictions: 200

   Prediction distribution:


In [24]:


submission = pd.DataFrame({
    'smiles': kaggle_test_df['smiles'].values,
    'stability': predictions
})

submission.to_csv(r'algor_used\kaggle_submission_ensemble.csv', index=False)

print(f" Saved: kaggle_submission_ensemble.csv")
print(f"\n   First 10 predictions:")
print(submission.head(10).to_string(index=False))


 Saved: kaggle_submission_ensemble.csv

   First 10 predictions:
                                                       smiles stability
[SiH2]1cccc1-c1ccc(-c2cnc(-c3nccc4nsnc34)c3nsnc23)c2c[nH]cc12       low
      c1[nH]cc2c1c(cc1c3ccccc3c3c4ccccc4oc3c21)-c1nccc2nsnc12      high
   c1cc2c(scc2o1)-c1ccc(o1)-c1ccc(-c2cccc3c[nH]cc23)c2ccccc12      high
       c1cc2csc(-c3cc4c(s3)c3cnc5ccc6ccccc6c5c3c3ccccc43)c2s1      high
                             C1cccc1-c1ccc(-c2ccccc2)c2nsnc12    medium
                 C1C=c2c3cc[se]c3c3sc4C=C(Cc4c3c2=C1)c1ccccn1      high
          c1[nH]cc2c1ccc1sc3c4ccc(cc4[se]c3c21)-c1cncc2nsnc12      high
                      c1c[nH]c(c1)-c1cc2sc3c4ncccc4[se]c3c2s1      high
    c1cc2csc(-c3ccc([nH]3)-c3cnc(-c4nccc5nsnc45)c4nsnc34)c2o1       low
              C1c2cc(-c3ncncn3)c3c[nH]cc3c2-c2[nH]c3cccnc3c12       low


In [25]:

print(f"LR F1:     {lr_f1:.4f}")
print(f"RF F1:     {rf_f1:.4f}")
print(f"Voting F1: {voting_f1:.4f}")


agreement = (y_pred_lr == y_pred_rf).mean()
print(f"Agreement: {agreement:.2%}")

LR F1:     0.9514
RF F1:     0.9352
Voting F1: 0.9514
Agreement: 98.39%


Thats why using LR or Voting made absolutely no difference, LR and RF are in 98% agreement, disagreed only on one sample and obviously voting decided to use LR's prediction